# N124 · 位运算与整数表示

**目标：** 在明确位宽下操作二进制并区分Python整数语义。

**先修：** N002, N005, N013。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 与或异或非；移位；补码；lowbit；清最低1；popcount；掩码。

**配套讲解来源：** [同名逐章意见](../../comment/124_bits_integer_representation.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这章练的是"把整数看成二进制位串来操作"的基本功，同时专门厘清 Python 整数和 C 语言 32 位整数在语义上的三个坑。围绕三个经典小问题展开：

1. 数一个整数二进制里有几个 1（popcount）。例子：n = 44，二进制是 00101100，里面有 3 个 1（对应 32+8+4），所以答案是 3。
2. 把一个 32 位无符号整数的二进制位整个翻转（reverse bits）。例子：n = 1（二进制低 32 位是 000...001），翻转后是 100...000，即 2³¹ = 2147483648。
3. 用"手算竖式"的思路把两个二进制字符串相加。例子：a = '11'，b = '1'，输出 '100'——最低位 1+1 得 0 进 1，进位再和次低位 1 相加又得 0 进 1，最后落下一个 1。

这一章比算法更重要的是"语义坑"：Python 的整数没有 32 位上限（不会溢出，但位翻转题必须自己截取低 32 位）；Python 的负数在位运算时按无限长的补码理解（-1 相当于无穷个 1），直接套"清 1 到零"的循环会死循环。本章代码用显式防御和掩码把这些边界全部写明。

## 2. 基础知识：先读懂这些词

- **二进制位（bit）与位宽**：整数在计算机里是一排 0/1。位宽是这排位的个数，比如 32 位机上的 int 就是 32 个 bit。Python 整数没有固定位宽，它想要多少位就有多少位（任意精度）。
- **与/或/异或/非（&、|、^、~）**：四种按位操作。与：两位都是 1 才是 1；或：有 1 就是 1；异或：两位不同才是 1；非：每位取反。本章重点用与（做掩码、清位）。
- **移位（<<、>>）**：n<<1 是整体左移一位（相当于乘 2，低位补 0）；n>>1 是右移一位（相当于除 2 取整，挤掉最低位）。本章的反转和进位都靠移位。
- **补码（two's complement）**：定点表示负数的标准方案，规则是"按位取反再加一"。它的好处是加法器不用区分正负。C 语言里 -1 的 32 位补码是全 1；Python 里 -1 概念上是"无穷多个 1"，这叫无限符号扩展。
- **掩码（mask）**：一个特制的 0/1 串，用来"套"出或"截"出我们想要的位。本章最常用的掩码是 (1<<32)−1，即低 32 位全 1，任何数和它做与运算就只保留低 32 位。
- **lowbit（最低位的 1）**：n & (−n) 得到"只保留 n 最低那个 1"的数。例如 44 & (−44) = 4。它是树状数组等结构的好朋友（本章知识点清单里点名了它）。
- **清最低位的 1（n & (n−1)）**：把 n 的二进制里最低的那个 1 抹掉、其他位不动。例如 44 (101100) & 43 (101011) = 40 (101000)。它是 popcount 快写法的关键。
- **popcount（置位计数）**：数二进制表示里 1 的个数。Python 3.10+ 自带 n.bit_count()，测试里就拿它当标准答案对拍。
- **carry（进位）**：二进制加法里"这一位相加满 2 就向高位进 1"。竖式加法的每一步就是"两位加进位，写余数、传进位"。

## 3. 思路推导：从小例子开始

- Step 1：先搞懂 n&(n−1) 为什么恰好删掉最低的 1。设 n 的最低 1 在第 k 位：n 写出来是 ...1000…0（第 k 位是 1、右边全是 0）。n−1 借位后变成 ...0111…1（第 k 位变 0、右边全变 1）。两者做与运算：第 k 位 1&0 = 0，右边 0&1 = 0，左边高位原样保留——结果就是"只少了那个 1"的 n。
- Step 2：手算一遍清位链。取 n = 44（00101100）。下面这张表就是 notebook “运行示例”部分 打印的内容：

| 十进制 | 二进制 | 清除最低1（n&(n−1)） |
|---|---|---|
| 44 | 00101100 | 00101000（=40） |
| 40 | 00101000 | 00100000（=32） |
| 32 | 00100000 | 00000000（=0） |

  44 → 40 → 32 → 0，正好三步，所以 popcount(44) = 3。循环次数就等于 1 的个数，因为每一步精确消灭一个 1。
- Step 3：位翻转怎么做。思路是"从最低位开始，一位一位摘下来，拼到输出的高位去"：每轮执行 out = (out<<1)|(n&1)，然后 n >>= 1。用 4 位数 1011 演示 4 轮：第 1 轮摘到 1，out=1；第 2 轮摘到 1，out=11；第 3 轮摘到 0，out=110；第 4 轮摘到 1，out=1101。输入 1011 变成输出 1101，顺序正好倒过来。32 位版本把循环跑 32 次即可，第 i 位的输入最终落在第 31−i 位。
- Step 4：为什么先要 n &= (1<<32)−1。题目的语义是"32 位无符号整数"，而 Python 的整数没有这个边界。如果传进来 −1（Python 里是无限个 1），不先截断就会错；先用掩码套一下，−1 变成 4294967295（低 32 位全 1），翻转后仍是全 1，语义就对了。
- Step 5：二进制字符串加法怎么做。模拟小学竖式：两个指针 i、j 从两串末尾（最低位）出发，carry 记录来自低位的进位。每一轮：carry 累加两个当前位（存在的才算），本轮写下的位是 carry & 1，往上传的进位是 carry >> 1。手算 '11' + '1'：
  - 第 1 轮：1+1 = 2，写 0，进 1；
  - 第 2 轮：1（'11' 的次低位）+ 进位 1 = 2，写 0，进 1；
  - 第 3 轮：只剩进位 1，写 1，进 0。
  - 逆序拼接得到 '100'。循环条件 `i>=0 or j>=0 or carry` 里那个 carry 保证了最后孤零零的进位也会被写出。

## 4. 核心代码：popcount

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def popcount(n):
    if n < 0:
        raise ValueError('popcount requires nonnegative integer')
    count = 0
    while n:
        n &= n - 1
        count += 1
    return count
```

### 逐段读懂代码

### 4.1 popcount：清 1 计数

```python
def popcount(n):
    if n < 0:
        raise ValueError('popcount requires nonnegative integer')
    count = 0
    while n:
        n &= n - 1
        count += 1
    return count
```

- `if n<0: raise ValueError(...)` 是第一道防线：负数在 Python 里按无限补码理解（−1 是无穷个 1），n&(n−1) 永远清不到 0（−1 & −2 = −2，−2 & −3 = −4，……），循环会一直转下去。所以负数干脆拒绝服务。
- `while n: n&=n-1; count+=1` 是紧凑写法：循环体"清掉最低的 1、计数加一"挤在一行。n 变成 0 就说明 1 全清完了。循环次数恰好等于 1 的个数，这正是 Step 2 的结论。

### 4.2 reverse_bits32：翻转低 32 位

```python
def reverse_bits32(n):
    n &= (1 << 32) - 1
    out = 0
    for _ in range(32):
        out = out << 1 | n & 1
        n >>= 1
    return out
```

- `n&=(1<<32)-1` 先用掩码把 n 截成低 32 位。(1<<32)−1 就是 32 个 1：1 左移 32 位是 1 后面 32 个 0，减 1 借位后变成低 32 位全 1。这一步同时处理了负数（无限符号扩展被砍掉）和超过 32 位的大数。
- `for _ in range(32): out=(out<<1)|(n&1); n>>=1` 每轮做两件事：把 out 的既有位整体左移腾出最低位，把 n 的最低位（n&1）接上去；然后 n 右移丢弃刚处理过的位。32 轮之后，原数的第 i 位恰好落在输出的第 31−i 位。
- 固定 32 轮，所以无论输入多大这个函数的工作量都是常数。

### 4.3 add_binary：二进制字符串竖式加法

```python
def add_binary(a, b):
    if any((c not in '01' for c in a + b)):
        raise ValueError('binary strings required')
    i, j = (len(a) - 1, len(b) - 1)
    carry = 0
    out = []
    while i >= 0 or j >= 0 or carry:
        if i >= 0:
            carry += ord(a[i]) - 48
            i -= 1
        if j >= 0:
            carry += ord(b[j]) - 48
            j -= 1
        out.append(str(carry & 1))
        carry >>= 1
    return ''.join(reversed(out)).lstrip('0') or '0'
```

- `if any(c not in '01' for c in a+b): raise ValueError(...)` 是输入体检：任何一个字符不是 '0'/'1' 就拒收。注意 a+b 在这里是字符串拼接，顺手把两个串一起检查了。
- `i,j=len(a)-1,len(b)-1` 两个指针都从最低位（字符串末尾）出发，因为二进制串是高位在前、低位在后，竖式要从低位算起。
- 循环体里 `ord(a[i])-48` 把字符 '0'/'1' 换成数字 0/1（ASCII 码 '0' 是 48）。carry 可以暂时累积到 2 甚至 3（本位 1 + 本位 1 + 低位进位 1），`carry&1` 取出该写在答案里的位，`carry>>=1` 把剩下的进位（0 或 1）传给下一轮。因为一位加一位加进位最多 3，右移一次必然归到 0 或 1，不会漏。
- `return ''.join(reversed(out)).lstrip('0') or '0'` 收尾有三件事：out 是从低位到高位攒的，先 reversed 再 join 变成高位在前；lstrip('0') 抹掉前导零（比如 '000'+'0' 会算出 '000'，要归一化）；`or '0'` 兜底"全零串被 strip 成空串"的极端情形，保证至少返回一个 '0'。

## 5. 分段实现：按顺序运行

**本章接口：** `popcount(n)`、`reverse_bits32(n)`、`add_binary(a, b)`

### popcount

In [1]:
def popcount(n):
    if n < 0:
        raise ValueError('popcount requires nonnegative integer')
    count = 0
    while n:
        n &= n - 1
        count += 1
    return count

### reverse_bits32

In [2]:
def reverse_bits32(n):
    n &= (1 << 32) - 1
    out = 0
    for _ in range(32):
        out = out << 1 | n & 1
        n >>= 1
    return out

### add_binary

In [3]:
def add_binary(a, b):
    if any((c not in '01' for c in a + b)):
        raise ValueError('binary strings required')
    i, j = (len(a) - 1, len(b) - 1)
    carry = 0
    out = []
    while i >= 0 or j >= 0 or carry:
        if i >= 0:
            carry += ord(a[i]) - 48
            i -= 1
        if j >= 0:
            carry += ord(b[j]) - 48
            j -= 1
        out.append(str(carry & 1))
        carry >>= 1
    return ''.join(reversed(out)).lstrip('0') or '0'

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

n=44; rows=[]
while n:
    nxt=n&(n-1); rows.append((n,format(n,'08b'),format(nxt,'08b'))); n=nxt
show_table(['十进制','二进制','清除最低1'],rows)

十进制,二进制,清除最低1
44,00101100,00101000
40,00101000,00100000
32,00100000,00000000


## 7. 正确性、适用条件与复杂度

n−1把最低1变为0，并把其右侧零变成1；与原数相与正好只删除那一位。反转循环每步把原数最低位接到输出末尾，32步后各位位置映射i→31−i。加法carry记录来自较低位的进位，保持部分和一致。

**代价：** popcount执行O(置位数)次整数位操作；任意精度整数的每次位操作成本随位宽增长。32位反转固定O(1)；字符串加法O(max(m,n))时间和输出空间。

### 展开理解

先说三个算法为什么对。

n&(n−1) 只删最低的 1：Step 1 已经逐位推过——减法的借位把"最低 1 变 0、其右侧 0 全变 1"，与运算后恰好只有那一位消失、高位纹丝不动。所以 popcount 每转一圈就消灭一个 1、也只消灭一个 1，转几圈就有几个 1，计数正确。

位翻转的对合性：循环每步把 n 的最低位接到 out 的末尾，等价于把第 i 位搬到第 31−i 位。你做两次翻转，第 i 位先到 31−i 又回到 i，所以"翻转的翻转等于原数"（测试正是这么验的）。

加法的进位不变量：每一轮处理完第 k 位时，carry 恰好等于"a 的低 k 位加 b 的低 k 位再除以 2^(k+1) 向下取整"——也就是说，写出的位串加上 carry×2^(k+1) 恒等于两个已处理部分之和。循环结束（两串耗尽且 carry 为 0）时，out 加 0 恰是完整和，没有丢失任何信息。

复杂度（结合具体规模感受）：

- popcount：循环次数 = 1 的个数，最坏 32 位数是 32 次、n 位整数是 O(n) 次。注意 Python 整数是任意精度，每个"&"在超大数上不是常数时间——两个一百万位的数相与，要动大约一百万位的工作量（练习 2 就是让你把这个说清楚）。
- reverse_bits32：固定 32 轮，工作量是常数 O(1)。
- add_binary：一轮处理一位，轮数 = max(len(a), len(b)) + 最后那点进位，时间 O(max(m,n))，输出串也这么长。两串各 10⁴ 位时也就一万多轮循环，瞬间完成。

## 8. 单元测试：每个用例在检查什么

```python
assert popcount(0) == 0 and popcount(7) == 3
```
前半条测零值边界（0 里没有 1，while 循环一次都不进，count 保持 0）；后半条测"全 1 串"：7 = 111，要清三次才到 0。

```python
assert add_binary('11', '1') == '100' and add_binary('000', '0') == '0'
```
前半条测进位传播：进位一路顶到最高位再溢出一位，考察 `or carry` 那个循环条件；后半条测前导零归一化：输入带着前导零，输出必须是 '0' 而不是 '000' 或空串，考的是 lstrip 加 or 的收尾组合。

```python
for n in range(1000):
    assert popcount(n) == n.bit_count()
    assert reverse_bits32(reverse_bits32(n)) == n
```
这是两路标准答案对拍：popcount 与 Python 自带的 bit_count 在 0–999 上逐一相等；位翻转做两次回到自身（对合性）。后者比"和手写反转比一次"更强——它同时验证了截断和移位方向都没偏。

```python
assert reverse_bits32(-1) == (1 << 32) - 1
```
这条专测负数语义：−1 是无限个 1，函数必须先被掩码截成低 32 位全 1，翻转后仍是全 1（4294967295）。它锁住"显式截取低 32 位"这个契约。

```python
for a in range(64):
    for b in range(64):
        assert add_binary(format(a, 'b'), format(b, 'b')) == format(a + b, 'b')
```
把 0–63 两两组合成 4096 对，先 format 成二进制串喂给 add_binary，再和整数直接相加的标准答案比。这覆盖了"长度不等、连续进位、双方全零"等情形。

```python
try:
    popcount(-1)
except ValueError:
    pass
else:
    raise AssertionError('negative domain must be rejected')
```
这条测防御契约：负数必须被 ValueError 拒绝，如果 popcount(-1) 竟然返回了值（意味着你能绕过检查），测试主动报错。

In [5]:
assert popcount(0) == 0 and popcount(7) == 3

assert add_binary('11', '1') == '100' and add_binary('000', '0') == '0'

for n in range(1000):
    assert popcount(n) == n.bit_count()
    assert reverse_bits32(reverse_bits32(n)) == n

assert reverse_bits32(-1) == (1 << 32) - 1

for a in range(64):
    for b in range(64):
        assert add_binary(format(a, 'b'), format(b, 'b')) == format(a + b, 'b')

try:
    popcount(-1)
except ValueError:
    pass
else:
    raise AssertionError('negative domain must be rejected')

print('N124: 所有本章断言通过')

N124: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 说明Python的~x不是有限位按位翻转结果。

**练习 2：** 解释大整数位操作不总是常数成本。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（说明 Python 的 ~x 不是有限位按位翻转）**：提示——你在 32 位的世界里期望 ~5 是"低 32 位取反"，但 Python 里 print(~5) 会得到 −6。原因是 Python 的整数按无限位补码理解，~x 的真身是 −x−1：5 是 ...000101，取反成 ...111010，这个无限串按补码读就是 −6。验证方向：把 ~x 与掩码 (1<<32)−1 做与运算，看结果是否等于 32 位意义下的按位取反（例如 ~5 & 0xFFFFFFFF = 4294967290，正是 0xFFFFFFFA）。写清楚"想要 32 位语义必须显式限定掩码"这个结论。

**练习 2（大整数位操作不总是常数成本）**：提示——C 里 int 固定 32 位，一次 & 是常数时间；Python 的整数要多少位有多少位，一次 & 的成本随位数增长。你可以构造 x = 1 << 1000000（一个一百万位的数），对比 popcount(x)（清一次 1）和 popcount(x−1)（要清一百万个 1）的耗时差异；再说明即使只做一次 n&(n−1)，两个百万位的数相与也要动约百万位。结论口径：本章说的 O(置位数) 指的是"整数位操作的次数"，每次操作本身的成本要按位宽另算。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
def popcount(n):
    if n < 0:
        raise ValueError('popcount requires nonnegative integer')
    count = 0
    while n:
        n &= n - 1
        count += 1
    return count

def reverse_bits32(n):
    n &= (1 << 32) - 1
    out = 0
    for _ in range(32):
        out = out << 1 | n & 1
        n >>= 1
    return out

def add_binary(a, b):
    if any((c not in '01' for c in a + b)):
        raise ValueError('binary strings required')
    i, j = (len(a) - 1, len(b) - 1)
    carry = 0
    out = []
    while i >= 0 or j >= 0 or carry:
        if i >= 0:
            carry += ord(a[i]) - 48
            i -= 1
        if j >= 0:
            carry += ord(b[j]) - 48
            j -= 1
        out.append(str(carry & 1))
        carry >>= 1
    return ''.join(reversed(out)).lstrip('0') or '0'

# 示例与回归测试
assert popcount(0) == 0 and popcount(7) == 3

assert add_binary('11', '1') == '100' and add_binary('000', '0') == '0'

for n in range(1000):
    assert popcount(n) == n.bit_count()
    assert reverse_bits32(reverse_bits32(n)) == n

assert reverse_bits32(-1) == (1 << 32) - 1

for a in range(64):
    for b in range(64):
        assert add_binary(format(a, 'b'), format(b, 'b')) == format(a + b, 'b')

try:
    popcount(-1)
except ValueError:
    pass
else:
    raise AssertionError('negative domain must be rejected')

print('N124: 所有本章断言通过')

N124: 所有本章断言通过


## 11. 代表题与迁移

- **191. Number of 1 Bits**：数无符号整数的 1 的个数，popcount 函数即题解本体。这题练的就是 n&(n−1) 清最低位 1 这招。
- **190. Reverse Bits**：翻转 32 位无符号整数的所有位，reverse_bits32 即题解。这题练的是"掩码截位 + 逐位摘拼"的套路。
- **338. Counting Bits**：对 0..n 每个数都数 1 的个数。这题练的是把本章技巧改成递推：i 的答案等于 i&(i−1) 的答案加一（清掉一个 1 再查表），一步到位。
- **67. Add Binary**：两个二进制字符串相加，add_binary 即题解。这题练的是 carry 的维护和"最后一位进位别丢"的细节。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。